In [1]:
trading_framework/
├─ config.py
├─ run_backtests.py                # main entrypoint
├─ data_paths.py                   # centralises file paths (Windows-friendly)
├─ core/
│  ├─ engine.py                    # sizing loop, P&L, turnover, IO
│  ├─ metrics.py                   # sharpe/sortino/mdd/calmar, MAE/MFE, etc.
│  └─ utils.py                     # common helpers (annualise, EWMA variance, etc.)
├─ strategies/
│  ├─ ewma.py                      # EWMA family (fast/slow, scalers)
│  └─ carry.py                     # Carry (near/far)
└─ outputs/                        # CSVs land here


SyntaxError: invalid character '├' (U+251C) (2588952648.py, line 2)

In [6]:
import os, numpy as np, pandas as pd

# ======== CONFIG (change per instrument) ========
IN_DIR   = r"C:\Users\loci_\Desktop\trading_webapp\DATA\all_input_files"
OUT_DIR  = r"C:\Users\loci_\Desktop\trading_webapp\DATA\all_output_files"
INSTRUMENT = "RX1_small.csv"     # e.g. "AD1_small.csv"
OUT_PREFIX = "RX1"               # e.g. "AD1"

TRADING_DAYS = 256
CAP = 20.0
DECAY_N = 36
ALPHA = 2 / (DECAY_N + 1)

# fast → scaler (slow = fast*4)
CROSSES = [
    (2, 10.6),
    (4, 7.5),
    (8, 5.3),
    (16, 3.75),
    (32, 2.65),
    (64, 1.87),
]

os.makedirs(OUT_DIR, exist_ok=True)

# ======== LOAD ========
path = os.path.join(IN_DIR, INSTRUMENT)
df = pd.read_csv(path)
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)

# be robust to px column case
px_col = "PX_CLOSE_1D" if "PX_CLOSE_1D" in df.columns else ("px_close_1d" if "px_close_1d" in df.columns else None)
if px_col is None:
    raise KeyError("Price column PX_CLOSE_1D not found.")
px = df[px_col].astype(float)

# ======== EWMA variance on net returns (Carver decay) ========
ret_net = px.diff()
var = np.full(len(df), np.nan)
arr = ret_net.to_numpy()
i0 = np.argmax(~np.isnan(arr)) if np.any(~np.isnan(arr)) else None
if i0 is not None:
    var[i0] = arr[i0]**2
    prev = var[i0]
    for i in range(i0+1, len(arr)):
        x = 0.0 if np.isnan(arr[i]) else arr[i]
        prev = ALPHA*(x*x) + (1-ALPHA)*prev
        var[i] = prev
vol = pd.Series(np.sqrt(var), index=df.index).replace(0.0, np.nan)

ret_pct = px.pct_change()

def compute_ewma_forecast(px, vol, fast, slow, scaler, cap=20.0):
    ewf = px.ewm(span=fast, adjust=False).mean()
    ews = px.ewm(span=slow, adjust=False).mean()
    raw = (ewf - ews) / vol
    return (raw * scaler).clip(-cap, cap)

results = []
for fast, scaler in CROSSES:
    slow = fast * 4
    fcast = compute_ewma_forecast(px, vol, fast, slow, scaler, CAP)
    raw_pnl = fcast.shift(1) * ret_pct
    sharpe = np.nan
    sd = raw_pnl.std()
    if sd and not np.isnan(sd) and sd != 0:
        sharpe = (raw_pnl.mean() / sd) * np.sqrt(TRADING_DAYS)
    label = f"{fast}d_{slow}d"
    print(f"{OUT_PREFIX}: {label:>9s} | scaler={scaler:>5.2f} | Sharpe={sharpe:>6.3f}")

    out = df[["Date", px_col]].copy()
    out.rename(columns={px_col: "PX_CLOSE_1D"}, inplace=True)
    out["forecast_ewma"] = fcast
    out.to_csv(os.path.join(OUT_DIR, f"{OUT_PREFIX}_EWMA_{label}.csv"), index=False)

    results.append((fast, slow, scaler, sharpe))

summary = pd.DataFrame(results, columns=["fast", "slow", "scaler", "sharpe"])
summary.to_csv(os.path.join(OUT_DIR, f"{OUT_PREFIX}_EWMA_summary.csv"), index=False)
print(f"\nSaved EWMA files for {OUT_PREFIX} to: {OUT_DIR}")


RX1:     2d_8d | scaler=10.60 | Sharpe= 0.053
RX1:    4d_16d | scaler= 7.50 | Sharpe=-0.567
RX1:    8d_32d | scaler= 5.30 | Sharpe=-1.250
RX1:   16d_64d | scaler= 3.75 | Sharpe=-1.345
RX1:  32d_128d | scaler= 2.65 | Sharpe=-1.156
RX1:  64d_256d | scaler= 1.87 | Sharpe=-0.999

Saved EWMA files for RX1 to: C:\Users\loci_\Desktop\trading_webapp\DATA\all_output_files
